# Simulacro de clasificación

Mismo dataset y mismo caso de OlimpiAlpes (ver `../practica_regresion/OlimpiAlpes.ipynb`), pero tratado como un problema de **clasificación binaria** en vez de regresión, para practicar qué cambia y qué se mantiene igual.

Este notebook sigue el mismo alcance que pide el enunciado de la Pregunta 2 (entendimiento, calidad, transformaciones, pipeline aplicado sobre test) — **no incluye entrenar/predecir con un modelo**, porque eso no es parte de lo que califica esa pregunta. Referencia teórica completa: sección "Si el caso es clasificación en vez de regresión" en `../guias_estudio/guia_codigo_y_decisiones.md`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../datos/Dataset_preparacion_atletica.csv')
data = df.copy()
data.shape

## Corrección de calidad (idéntica a la práctica de regresión)

Se reaplican las mismas correcciones ya investigadas y justificadas en `OlimpiAlpes.ipynb`: Género (M/F → Masculino/Femenino), Edad (error de unidad meses→años), Frecuencia de competencia (negativos inválidos → NaN), y duplicados exactos. La calidad de datos no cambia por tratarse de clasificación en vez de regresión.

In [ ]:
data['Género'] = data['Género'].replace({'M': 'Masculino', 'F': 'Femenino'})
data.loc[data['Edad'] > 60, 'Edad'] = data.loc[data['Edad'] > 60, 'Edad'] / 12
data.loc[data['Frecuencia de competencia'] < 0, 'Frecuencia de competencia'] = np.nan
data = data.drop_duplicates(subset=[c for c in data.columns if c != 'log_id'], keep='first')
data.shape

## Paso nuevo: crear el target binario

En un caso real de clasificación el target categórico ya vendría dado en los datos. Aquí lo derivamos de `Puntuación de rendimiento` (umbral en 88) solo para poder practicar clasificación con el mismo dataset que ya conocemos.

In [ ]:
data['Alto_rendimiento'] = (data['Puntuación de rendimiento'] >= 88).astype(int)

data['Alto_rendimiento'].value_counts()

In [ ]:
data['Alto_rendimiento'].value_counts(normalize=True) * 100
# Aquí SÍ aplica revisar el balance de clases -> esto NO existía en el punto 2 de la práctica de regresión

## EDA del target (cambia de forma respecto a regresión)

- Numérica vs target categórico → boxplot agrupado por clase (antes era scatter contra un target continuo).
- Categórica vs target categórico → crosstab (antes era boxplot contra un target continuo).

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=data, x='Alto_rendimiento', y='Horas de entrenamiento por semana')
plt.show()

In [ ]:
pd.crosstab(data['Nivel de experiencia'], data['Alto_rendimiento'], normalize='index')

## Separar X/y y split (con `stratify`)

`stratify=y` es nuevo respecto a la práctica de regresión: asegura que train y test mantengan la misma proporción de clases 0/1. No arregla el desbalance por sí solo, solo evita empeorarlo por azar en el split.

In [ ]:
X = data.drop(columns=['Puntuación de rendimiento', 'Alto_rendimiento', 'log_id'])
y = data['Alto_rendimiento']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Pipeline de preparación (idéntico al de la práctica de regresión)

La preparación de datos (imputación, escalado, encoding) no depende del modelo final — es exactamente el mismo `ColumnTransformer` que en `OlimpiAlpes.ipynb`.

In [ ]:
numeric_cols = ['Edad', 'IMC', 'Horas de entrenamiento por semana', 'Frecuencia cardíaca en reposo',
                'Ingesta de calorías', 'Frecuencia de competencia', 'Puntuación de preparación estratégica']
ordinal_cols = ['Nivel de experiencia', 'Indicador de establecimiento de objetivos']
nominal_cols = ['Género', 'Tipo de deporte']

assert len(numeric_cols) + len(ordinal_cols) + len(nominal_cols) == X_train.shape[1]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

ordinal_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[
        ['Principiante', 'Intermedio', 'Avanzado', 'Profesional'],
        ['No', 'Sí']
    ]))
])

nominal_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('ord', ordinal_transformer, ordinal_cols),
    ('nom', nominal_transformer, nominal_cols)
])

# fit_transform SOLO en train
X_train_prep = preprocessor.fit_transform(X_train)

# transform (sin fit) sobre test
X_test_prep = preprocessor.transform(X_test)

In [ ]:
X_test_prep_df = pd.DataFrame(X_test_prep, columns=preprocessor.get_feature_names_out())
display(X_test_prep_df.head())

## Nota

Si quisieras seguir hasta entrenar y evaluar un modelo (no lo pide esta pregunta del examen, pero es útil para entender el flujo completo y la interpretabilidad vía coeficientes), el siguiente paso sería:

```python
from sklearn.linear_model import LogisticRegression

modelo = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
```

Se deja solo como referencia en texto, no como celda ejecutable, para que el notebook refleje exactamente el mismo alcance que `OlimpiAlpes.ipynb`.